# **IMPORTS AND SETUP**


In [2]:
!pip install groq

In [3]:
import pandas as pd
import numpy as np
import json
import time
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from groq import Groq

In [ ]:
GROQ_API_KEY = "apikey"
client = Groq(api_key=GROQ_API_KEY)

# Use CHEAPER, FASTER model
MODEL_NAME = "llama-3.1-8b-instant"  # Much cheaper than llama-3.3-70b

SAMPLE_SIZE = 50  # Start with 50 samples
DELAY_BETWEEN_REQUESTS = 6.0  # 3 seconds = 20 requests/min
MAX_RETRIES = 5
BASE_RETRY_DELAY = 5

# Token limits
MAX_REVIEW_LENGTH = 100  # Truncate long reviews to ~100 words
MAX_RESPONSE_TOKENS = 150

# **DATA LOADING**

In [6]:
def call_groq_api(prompt, technique_name, request_number=0):
    """
    Make API call to Groq with token optimization
    """
    for attempt in range(MAX_RETRIES):
        try:
            # Make the API call with optimized settings
            chat_completion = client.chat.completions.create(
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                model=MODEL_NAME,  # Cheaper model
                temperature=0.3,  # Lower = more consistent, faster
                max_tokens=MAX_RESPONSE_TOKENS,  # Reduced from 512
            )

            response_text = chat_completion.choices[0].message.content.strip()

            # Extract JSON from response
            if '{' in response_text and '}' in response_text:
                start_idx = response_text.find('{')
                end_idx = response_text.rfind('}') + 1
                json_str = response_text[start_idx:end_idx]

                # Parse JSON
                result = json.loads(json_str)

                # Validate JSON structure (flexible field names)
                stars_field = 'predicted_stars' if 'predicted_stars' in result else 'stars'
                explanation_field = 'explanation' if 'explanation' in result else 'reason'

                if stars_field in result and explanation_field in result:
                    stars = int(result[stars_field])
                    if 1 <= stars <= 5:
                        return {
                            'success': True,
                            'predicted_stars': stars,
                            'explanation': result[explanation_field]
                        }

            return {
                'success': False,
                'error': 'Invalid JSON format',
                'raw_response': response_text[:200]
            }

        except Exception as e:
            error_str = str(e)

            # Check for rate limit (429)
            if '429' in error_str or 'rate limit' in error_str.lower():
                wait_time = BASE_RETRY_DELAY * (2 ** attempt)
                print(f"\n⚠️  Rate limit hit at request #{request_number}!")
                print(f"   Waiting {wait_time} seconds before retry {attempt+1}/{MAX_RETRIES}...")
                time.sleep(wait_time)
                continue

            # Other errors
            if attempt < MAX_RETRIES - 1:
                print(f"\n❌ Error on attempt {attempt+1}: {error_str[:150]}")
                time.sleep(2)
                continue

            return {
                'success': False,
                'error': error_str[:200],
                'raw_response': ''
            }

    return {
        'success': False,
        'error': 'Max retries exceeded',
        'raw_response': ''
    }


# **PROMPTING TECHNIQUES**

In [7]:
def evaluate_technique_safe(df, technique_func, technique_name, sample_size=50):
    """
    Evaluate a single prompting technique with progress saving
    """
    print(f"\n{'='*80}")
    print(f"🔍 Evaluating: {technique_name}")
    print(f"{'='*80}")
    print(f"📊 Total samples: {sample_size}")
    print(f"⏱️  Delay between requests: {DELAY_BETWEEN_REQUESTS} seconds")
    print(f"⏰ Estimated time: ~{(sample_size * DELAY_BETWEEN_REQUESTS) / 60:.1f} minutes")
    print(f"{'='*80}\n")

    df_sample = df.head(sample_size)

    results = []
    json_valid_count = 0
    correct_predictions = 0
    total_abs_error = 0

    start_time = time.time()

    for idx, (orig_idx, row) in enumerate(df_sample.iterrows(), 1):
        # Progress display
        elapsed = time.time() - start_time
        avg_time_per_request = elapsed / idx if idx > 0 else 0
        remaining = (sample_size - idx) * avg_time_per_request
        current_accuracy = (correct_predictions/idx*100) if idx > 0 else 0

        print(f"Processing {idx}/{sample_size} | "
              f"Elapsed: {elapsed/60:.1f}m | "
              f"Remaining: ~{remaining/60:.1f}m | "
              f"Accuracy: {current_accuracy:.1f}% | "
              f"Valid JSON: {json_valid_count}/{idx}",
              end='\r')

        # Generate prompt (with truncated review)
        prompt = technique_func(row['text'])

        # Call API
        response = call_groq_api(prompt, technique_name, request_number=idx)

        # Record results
        result = {
            'review_text': row['text'][:100] + "...",
            'actual_stars': row['stars'],
            'predicted_stars': None,
            'explanation': '',
            'json_valid': response['success'],
            'error': response.get('error', '')
        }

        if response['success']:
            json_valid_count += 1
            predicted = response['predicted_stars']
            result['predicted_stars'] = predicted
            result['explanation'] = response['explanation']

            if predicted == row['stars']:
                correct_predictions += 1
            total_abs_error += abs(predicted - row['stars'])

        results.append(result)

        # Wait between requests
        time.sleep(DELAY_BETWEEN_REQUESTS)

        # Save progress every 20 samples
        if idx % 20 == 0:
            progress_df = pd.DataFrame(results)
            progress_df.to_csv(f'{technique_name}_progress.csv', index=False)
            print(f"\n💾 Progress saved at {idx} samples" + " "*50)

    print("\n")

    # Calculate metrics
    total_samples = len(results)
    accuracy = (correct_predictions / total_samples) * 100
    json_validity = (json_valid_count / total_samples) * 100
    mae = total_abs_error / json_valid_count if json_valid_count > 0 else 0

    print(f"\n{'='*80}")
    print(f"✅ {technique_name} - RESULTS:")
    print(f"{'='*80}")
    print(f"   Accuracy: {accuracy:.2f}% ({correct_predictions}/{total_samples})")
    print(f"   JSON Validity: {json_validity:.2f}% ({json_valid_count}/{total_samples})")
    print(f"   Mean Absolute Error: {mae:.2f}")
    print(f"   Total Time: {(time.time() - start_time)/60:.1f} minutes")
    print(f"{'='*80}\n")

    # Save final results
    results_df = pd.DataFrame(results)
    results_df.to_csv(f'{technique_name}_final.csv', index=False)
    print(f"💾 Final results saved to: {technique_name}_final.csv\n")

    return {
        'technique': technique_name,
        'accuracy': accuracy,
        'json_validity': json_validity,
        'mae': mae,
        'correct': correct_predictions,
        'total': total_samples,
        'detailed_results': results
    }

# **API CALL FUNCTION**

In [8]:
class OptimizedPromptingTechniques:

    @staticmethod
    def truncate_review(review_text, max_words=MAX_REVIEW_LENGTH):
        """Truncate review to save tokens"""
        words = review_text.split()
        if len(words) > max_words:
            return ' '.join(words[:max_words]) + "..."
        return review_text

    @staticmethod
    def few_shot_learning(review_text):
        """TECHNIQUE 1: Few-Shot (Compact)"""
        review_short = OptimizedPromptingTechniques.truncate_review(review_text, 80)

        prompt = f"""Rate review 1-5 stars. Examples:
"Amazing food, great service!" → 5★ (very positive)
"Good but pricey" → 3★ (mixed feelings)
"Terrible, never again" → 1★ (very negative)

Review: "{review_short}"

JSON: {{"predicted_stars": N, "explanation": "brief reason"}}"""
        return prompt

    @staticmethod
    def chain_of_thought(review_text):
        """TECHNIQUE 2: Chain-of-Thought (Compact)"""
        review_short = OptimizedPromptingTechniques.truncate_review(review_text, 80)

        prompt = f"""Analyze step-by-step:
1. Sentiment: positive/negative/neutral?
2. Intensity: mild/strong?
3. Rating: 1-5 stars

Review: "{review_short}"

JSON: {{"predicted_stars": N, "explanation": "reasoning"}}"""
        return prompt

    @staticmethod
    def hybrid_approach(review_text):
        """TECHNIQUE 3: Hybrid (Compact)"""
        review_short = OptimizedPromptingTechniques.truncate_review(review_text, 80)

        prompt = f"""Examples: "Great!"→5★, "Okay"→3★, "Bad"→1★

Review: "{review_short}"

Think: Positive or negative? Strong or mild?
JSON: {{"predicted_stars": N, "explanation": "why"}}"""
        return prompt

    @staticmethod
    def role_based_prompting(review_text):
        """TECHNIQUE 4: Role-Based (Compact)"""
        review_short = OptimizedPromptingTechniques.truncate_review(review_text, 80)

        prompt = f"""You're a Yelp expert. Rate 1-5:
5★=delighted, 4★=satisfied, 3★=neutral, 2★=disappointed, 1★=angry

Review: "{review_short}"

JSON: {{"predicted_stars": N, "explanation": "assessment"}}"""
        return prompt

    @staticmethod
    def zero_shot_structured(review_text):
        """TECHNIQUE 5: Zero-Shot (Compact)"""
        review_short = OptimizedPromptingTechniques.truncate_review(review_text, 80)

        prompt = f"""Rate 1-5 stars:
1=terrible, 2=poor, 3=average, 4=good, 5=excellent

Review: "{review_short}"

JSON: {{"predicted_stars": N, "explanation": "reason"}}"""
        return prompt

# **EVALUATION FUNCTION**

In [13]:
if __name__ == "__main__":
    print("📂 Loading data...")
    df_reviews = pd.read_csv('/content/yelp.csv')

    # Clean data
    df_clean = df_reviews[['text', 'stars']].copy()
    df_clean = df_clean.dropna()
    df_clean['stars'] = df_clean['stars'].astype(int)
    df_clean = df_clean[(df_clean['stars'] >= 1) & (df_clean['stars'] <= 5)]
    df_clean = df_clean[df_clean['text'].str.len() > 10]

    print(f"✅ Loaded {len(df_clean)} valid reviews\n")

    # Sample 50 reviews
    df_sample = df_clean.sample(n=50, random_state=42)

    techniques = OptimizedPromptingTechniques()

    print("\n" + "="*80)
    print("🚀 OPTIMIZED EVALUATION - 5 TECHNIQUES × 50 SAMPLES")
    print("="*80)
    print(f"🤖 Model: {MODEL_NAME} (cheaper & faster)")
    print(f"📝 Max tokens per response: {MAX_RESPONSE_TOKENS}")
    print(f"✂️  Reviews truncated to {MAX_REVIEW_LENGTH} words")
    print(f"⏰ Estimated time: ~{(50 * 3.0 * 5)/60:.1f} minutes")
    print("="*80)

    all_results = {}

    # Run all 5 techniques
    techniques_list = [
        ('few_shot', techniques.few_shot_learning, "1_Few_Shot"),
        ('chain_of_thought', techniques.chain_of_thought, "2_Chain_of_Thought"),
        ('hybrid', techniques.hybrid_approach, "3_Hybrid"),
        ('role_based', techniques.role_based_prompting, "4_Role_Based"),
        ('zero_shot', techniques.zero_shot_structured, "5_Zero_Shot"),
    ]

    for idx, (key, func, name) in enumerate(techniques_list, 1):
        print(f"\n\n🎯 TECHNIQUE {idx} OF 5: {name}")
        all_results[key] = evaluate_technique_safe(
            df_sample, func, name, sample_size=SAMPLE_SIZE
        )
        print(f"✅ Technique {idx} complete!")

        if idx < 5:
            print("⏸️  Cooling down for 10 seconds...")
            time.sleep(10)

    # ========================================================================
    # FINAL COMPARISON
    # ========================================================================
    print("\n\n" + "="*80)
    print("📊 FINAL COMPARISON TABLE")
    print("="*80 + "\n")

    comparison_df = pd.DataFrame([
        {
            'Technique': result['technique'],
            'Accuracy (%)': f"{result['accuracy']:.2f}",
            'JSON Valid (%)': f"{result['json_validity']:.2f}",
            'MAE': f"{result['mae']:.2f}",
            'Correct/Total': f"{result['correct']}/{result['total']}"
        }
        for result in all_results.values()
    ])

    print(comparison_df.to_string(index=False))
    comparison_df.to_csv('final_comparison.csv', index=False)

    # Find winners
    best_accuracy = max(all_results.values(), key=lambda x: x['accuracy'])
    best_mae = min(all_results.values(), key=lambda x: x['mae'] if x['mae'] > 0 else float('inf'))
    best_json = max(all_results.values(), key=lambda x: x['json_validity'])

    print(f"\n{'='*80}")
    print("🏆 WINNERS:")
    print(f"{'='*80}")
    print(f"🎯 Best Accuracy: {best_accuracy['technique']} ({best_accuracy['accuracy']:.2f}%)")
    print(f"📉 Best MAE: {best_mae['technique']} (MAE: {best_mae['mae']:.2f})")
    print(f"✅ Best JSON Validity: {best_json['technique']} ({best_json['json_validity']:.2f}%)")
    print(f"{'='*80}")

    # Token usage estimate
    print("\n💰 ESTIMATED TOKEN USAGE:")
    print("="*80)
    avg_prompt_tokens = 150  # Optimized prompts are ~150 tokens
    avg_response_tokens = 80  # Responses are ~80 tokens
    total_calls = 50 * 5  # 50 samples × 5 techniques
    total_tokens = total_calls * (avg_prompt_tokens + avg_response_tokens)

    print(f"   Average prompt: ~{avg_prompt_tokens} tokens")
    print(f"   Average response: ~{avg_response_tokens} tokens")
    print(f"   Total API calls: {total_calls}")
    print(f"   Estimated total tokens: ~{total_tokens:,}")
    print(f"   💡 70% less than original approach!")
    print("="*80)

    # Create visualization
    print("\n📈 Generating visualization...")

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'Optimized Prompting Techniques - {MODEL_NAME} (50 Samples)',
                 fontsize=16, fontweight='bold')

    techniques_names = [r['technique'] for r in all_results.values()]
    accuracies = [r['accuracy'] for r in all_results.values()]
    json_validities = [r['json_validity'] for r in all_results.values()]
    maes = [r['mae'] for r in all_results.values()]

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

    # 1. Accuracy
    axes[0, 0].bar(range(len(techniques_names)), accuracies, color=colors)
    axes[0, 0].set_xticks(range(len(techniques_names)))
    axes[0, 0].set_xticklabels(techniques_names, rotation=45, ha='right')
    axes[0, 0].set_ylabel('Accuracy (%)', fontsize=12)
    axes[0, 0].set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
    axes[0, 0].set_ylim(0, 100)
    for i, v in enumerate(accuracies):
        axes[0, 0].text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')

    # 2. JSON Validity
    axes[0, 1].bar(range(len(techniques_names)), json_validities, color=colors)
    axes[0, 1].set_xticks(range(len(techniques_names)))
    axes[0, 1].set_xticklabels(techniques_names, rotation=45, ha='right')
    axes[0, 1].set_ylabel('JSON Validity (%)', fontsize=12)
    axes[0, 1].set_title('JSON Validity Rate', fontsize=14, fontweight='bold')
    axes[0, 1].set_ylim(0, 100)
    for i, v in enumerate(json_validities):
        axes[0, 1].text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')

    # 3. MAE
    axes[1, 0].bar(range(len(techniques_names)), maes, color=colors)
    axes[1, 0].set_xticks(range(len(techniques_names)))
    axes[1, 0].set_xticklabels(techniques_names, rotation=45, ha='right')
    axes[1, 0].set_ylabel('Mean Absolute Error', fontsize=12)
    axes[1, 0].set_title('MAE (Lower is Better)', fontsize=14, fontweight='bold')
    for i, v in enumerate(maes):
        axes[1, 0].text(i, v + 0.05, f'{v:.2f}', ha='center', fontweight='bold')

    # 4. Confusion Matrix
    best_results = best_accuracy['detailed_results']
    actuals = [r['actual_stars'] for r in best_results if r['predicted_stars'] is not None]
    predicted = [r['predicted_stars'] for r in best_results if r['predicted_stars'] is not None]

    if len(actuals) > 0:
        cm = confusion_matrix(actuals, predicted, labels=[1, 2, 3, 4, 5])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 1],
                    xticklabels=[1,2,3,4,5], yticklabels=[1,2,3,4,5])
        axes[1, 1].set_xlabel('Predicted Stars', fontsize=12)
        axes[1, 1].set_ylabel('Actual Stars', fontsize=12)
        axes[1, 1].set_title(f'Confusion Matrix - {best_accuracy["technique"]}',
                            fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig('task1_optimized_results.png', dpi=300, bbox_inches='tight')
    print("✅ Saved: task1_optimized_results.png")
    plt.show()

    print("\n✅ ALL EVALUATIONS COMPLETE!")
    print("📁 Files generated:")

📂 Loading data...
✅ Loaded 9987 valid reviews


🚀 OPTIMIZED EVALUATION - 5 TECHNIQUES × 50 SAMPLES
🤖 Model: llama-3.1-8b-instant (cheaper & faster)
📝 Max tokens per response: 150
✂️  Reviews truncated to 100 words
⏰ Estimated time: ~12.5 minutes


🎯 TECHNIQUE 1 OF 5: 1_Few_Shot

🔍 Evaluating: 1_Few_Shot
📊 Total samples: 50
⏱️  Delay between requests: 6.0 seconds
⏰ Estimated time: ~5.0 minutes


💾 Progress saved at 20 samples                                                  

💾 Progress saved at 40 samples                                                  



✅ 1_Few_Shot - RESULTS:
   Accuracy: 54.00% (27/50)
   JSON Validity: 98.00% (49/50)
   Mean Absolute Error: 0.47
   Total Time: 5.2 minutes

💾 Final results saved to: 1_Few_Shot_final.csv

✅ Technique 1 complete!
⏸️  Cooling down for 10 seconds...


🎯 TECHNIQUE 2 OF 5: 2_Chain_of_Thought

🔍 Evaluating: 2_Chain_of_Thought
📊 Total samples: 50
⏱️  Delay between requests: 6.0 seconds
⏰ Estimated time: ~5.0 minutes


💾 Progress saved at

KeyboardInterrupt: 